<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 6D · Tune Your Range: Experiment Until It's Honest and Narrow
### You pick the settings. 2025 judges them. 2026 gets one look. The winner goes live.

*Lab 6, notebook 4 of 4 · Colab · your energy project on Google Cloud*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Lab6D_Tune_Your_Range.ipynb)

In 6C you shipped a quantile forest with settings Dr. Dave picked: `n_estimators=100, min_samples_leaf=10`. Nobody
checked if those are good. Tonight **you** find out, the way real forecasters do: one experiment after another, every
one scored the same way, every one written down.

Your final project score is the **CRPS** of your live range (Nov 21 to Dec 1). Lab 5 tuned for MAE, one number per
hour. A setting that wins on MAE doesn't have to win on CRPS, because CRPS also judges the **range**. So tonight you
tune for CRPS.

| Part | Time | What you do |
|---|---|---|
| **0. Setup** | 5 min | Data, the HRRR archive, the tools from 6C |
| **1. Three periods, one rule** | 5 min | Train on 2023-2024, tune on 2025, look at 2026 **once** |
| **2. The experiment machine** | 5 min | `try_setting()`: one call = one experiment, scored like the leaderboard, written in the log |
| **3. Experiment 1: leaf size** | 10 min | The knob that sets how wide the range is |
| **4. Experiment 2: features per split** | 5 min | `max_features`: less is sometimes more |
| **5. Experiment 3: two knobs at once** | 10 min | A small grid and a heatmap |
| **6. Experiment 4: the range knobs** | 5 min | How many HRRR runs, and how much to widen |
| **7. Is it real?** | 5 min | Seed noise: is your gain bigger than luck? |
| **8. Your turn: experiment** | 15+ min | More trees, training on forecast weather, your own idea |
| **9. The log and the winner** | 5 min | Pick the winner with a rule, not a feeling |
| **10. One look at 2026** | 5 min | Your winner vs 6C's setting on the test year: your before and after |
| **11. Ship it** | 10 min | Refit, upload, run `forecast-24h`, check the tuned range is live |
| **12. Your dashboard on Cloud Run** | 10 min | Deploy your own public page: your live range, your score, every model you shipped |
| **13. Questions and submit** | 5 min | Questions 13 to 18 |

**This is the toil part of the lab.** Some experiments will lose. That's the point: a "this didn't help, and here's
the proof" is a real result, and it's exactly what the final project report asks for (section 5, one improvement).

# Part 0 · Setup (5 min)

In [ ]:
!pip install -q scikit-learn==1.6.1 quantile-forest==1.4.2

import io
import json
import os
import time
import warnings
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from pandas.tseries.holiday import USFederalHolidayCalendar
from quantile_forest import RandomForestQuantileRegressor

warnings.filterwarnings("ignore")
print("scikit-learn", sklearn.__version__, "(production uses 1.6.1)")
SEED = 42
LOCAL = "America/New_York"
DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])
archive = pd.read_csv(DATA + "bdl_hrrr_asissued_2023_2026.csv.gz", parse_dates=["init_utc", "hour_utc"])
print(len(df), "hours of weather and load |", archive["init_utc"].nunique(), "HRRR runs in the archive")

Your settings (the same as Lab 5 and 6C). Leave them alone for **demo mode**: everything runs except the upload in
Part 11.

In [ ]:
PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"     # e.g. "energy-dww05002-f26"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"    # usually the same as your project ID
NETID = "abc12345"                        # your UConn NetID, lower case

The tools from 6C, word for word: `make_features` (production's inputs), the leaderboard's `crps`, the HRRR runs
as they were really issued, and the 20 percentiles your `forecast-24h` pools. Run it and move on.

In [ ]:
WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]
READY_HOURS = 2                           # a run is published about 1 h 45 min after it starts
N_RUNS = 8
POOL_QUANTILES = list(np.round(np.arange(0.025, 0.976, 0.05), 3))     # 20 percentiles, 5% of the chances each


def make_features(hours_utc, weather, load):
    """One row of model inputs per hour (word for word from Lab 5)."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert("America/New_York")
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)
    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]


def crps(actual, low, expected, high):
    """The leaderboard's CRPS for each hour (MW, lower is better), from 6B."""
    actual = np.asarray(actual, dtype=float)
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)
    outside = np.clip(low - actual, 0, None) + np.clip(actual - high, 0, None)
    return (0.5 * np.abs(actual - np.asarray(expected, dtype=float)) + 0.1 * (high - low) + 1 * outside) / 1.5


def score_range(low, expected, high, actual):
    """CRPS, coverage and width over the hours given."""
    actual = np.asarray(actual, dtype=float)
    inside = (actual >= low) & (actual <= high)
    return {"CRPS (MW)": round(float(np.mean(crps(actual, low, expected, high))), 1),
            "inside range (%)": round(100 * float(np.mean(inside)), 1),
            "average width (MW)": round(float(np.mean(high - low)))}


def hrrr_runs_as_issued(target_hours, hours_ahead, n_runs=N_RUNS):
    """A list of n_runs weather tables (newest run first) for the target hours, as you'd have had them
    hours_ahead hours earlier (from 6C). Hours a run doesn't cover are NaN."""
    issued = target_hours - pd.Timedelta(hours=hours_ahead)
    newest = (issued - pd.Timedelta(hours=READY_HOURS)).floor("6h")
    tables = []
    for k in range(n_runs):
        wanted = pd.DataFrame({"init_utc": newest - pd.Timedelta(hours=6 * k), "hour_utc": target_hours})
        found = wanted.merge(archive, on=["init_utc", "hour_utc"], how="left")
        tables.append(pd.DataFrame(found[WEATHER].values, index=target_hours, columns=WEATHER))
    return tables


def percentiles_on_runs(forest, inputs):
    """The forest's 20 percentiles for every hour and every run: hours x runs x 20 (NaN = run doesn't cover the hour)."""
    out = np.full((len(inputs[0]), len(inputs), len(POOL_QUANTILES)), np.nan)
    for k in range(len(inputs)):
        ok = inputs[k][WEATHER].notna().all(axis=1).values
        if ok.any():
            out[ok, k, :] = forest.predict(inputs[k][ok], quantiles=POOL_QUANTILES)
    return out


def pooled(q, n_runs, widen):
    """Exactly what forecast-24h does every hour: pool the newest n_runs runs, read off the 10th, 50th and 90th
    percentile, then stretch low and high away from the expected value by range_widen."""
    pile = q[:, :n_runs, :].reshape(len(q), -1)
    low, expected, high = np.nanpercentile(pile, [10, 50, 90], axis=1, method="hazen")
    low = expected - widen * (expected - low)
    high = expected + widen * (high - expected)
    return low, expected, high


print("Tools ready")

# Part 1 · Three periods, one rule (5 min)
Lab 5 had two periods: train and test. Tuning needs **three**:

| Period | Years | Used for |
|---|---|---|
| **Train** | 2023 and 2024 | fitting every forest you try |
| **Validation** | 2025 | scoring every experiment: this is where you tune |
| **Test** | 2026 (to October 15) | **one look**, at the very end, for your before and after |

**The rule: never pick a setting by its 2026 score.** Try 30 settings and keep the one with the best 2026 CRPS, and
part of why it won is luck on those exact hours. Its 2026 score then looks better than it will really be live: you
tuned on the test. The leaderboard is a fresh test nobody has seen, so it would catch you. Tune on 2025. Look at 2026
once.

Every experiment is scored the way the leaderboard scores you: the **pooled HRRR range, made 12 hours ahead**, with the
runs as they were really issued (6C Part 2). To keep each experiment fast, it scores **every 3rd hour** of 2025: about
2,900 hours, plenty for an average.

In [ ]:
X_all = make_features(df.index, df, df["load_mw"]).iloc[25:]
y_all = df["load_mw"].loc[X_all.index]
years = X_all.index.tz_convert(LOCAL).year

TRAIN_YEARS = [2023, 2024]
VALID_YEAR = 2025
TEST_YEAR = 2026
VALID_AHEAD = 12                          # score ranges made 12 hours ahead, like 6C
VALID_EVERY = 3                           # every 3rd hour of 2025

in_train = years.isin(TRAIN_YEARS)
valid_hours = X_all.index[years == VALID_YEAR][::VALID_EVERY]
y_valid = y_all.loc[valid_hours]
valid_runs = hrrr_runs_as_issued(valid_hours, VALID_AHEAD)
valid_inputs = []
for k in range(N_RUNS):
    valid_inputs.append(make_features(valid_hours, valid_runs[k], df["load_mw"]))

periods = pd.DataFrame({
    "period": ["train", "validation", "test"],
    "years": ["2023-2024", str(VALID_YEAR), str(TEST_YEAR) + " (to Oct 15)"],
    "hours": [int(in_train.sum()), len(valid_hours), int((years == TEST_YEAR).sum())],
    "used for": ["fitting", "scoring every experiment (every 3rd hour)", "one look, at the end"]})
periods

# Part 2 · The experiment machine (5 min)
One experiment = one **settings** dictionary. `try_setting(settings, note)` does everything:

1. Fits a quantile forest on 2023-2024 with your forest settings (`n_estimators`, `min_samples_leaf`, `max_features`,
   `seed`) on the weather in `trained_on`.
2. Runs it on every HRRR run for every validation hour, 12 hours ahead.
3. Pools the newest `hrrr_runs` runs and widens by `range_widen`, exactly like `forecast-24h`.
4. Scores CRPS, coverage and width, and **writes one row in the experiment log**.

It remembers every forest it has fit, so changing only `hrrr_runs` or `range_widen` is instant.

**The production budget.** Your forest is a file that `forecast-24h` downloads and loads every hour. Small leaves make
big files: a forest with one training hour per leaf is about 80 MB. The log estimates the **shipped** file (refit on
all the data, Part 11). Keep it **under 50 MB**, or your function gets slow and can run out of memory.

In [ ]:
BUDGET_MB = 50
SETTING_NAMES = ["min_samples_leaf", "max_features", "n_estimators", "seed", "trained_on", "hrrr_runs", "range_widen"]
FOREST_NAMES = ["min_samples_leaf", "max_features", "n_estimators", "seed", "trained_on"]
EXPERIMENTS = []                          # the experiment log: one dictionary per experiment
forests = {}                              # every forest fit so far, so range knobs are instant


def training_table(trained_on, hours):
    """The training inputs for these hours: the weather that really happened ("observed weather", like Lab 5 and 6C),
    or the newest HRRR forecast issued 12 hours before each hour ("HRRR 12h forecast", Part 8 Challenge B)."""
    if trained_on == "observed weather":
        X = X_all.loc[hours]
    else:
        X = make_features(hours, hrrr_runs_as_issued(hours, 12, 1)[0], df["load_mw"])
        X = X[X[WEATHER].notna().all(axis=1)]
    return X, y_all.loc[X.index]


def fit_forest(settings, X, y):
    forest = RandomForestQuantileRegressor(n_estimators=settings["n_estimators"],
                                           min_samples_leaf=settings["min_samples_leaf"],
                                           max_features=settings["max_features"],
                                           random_state=settings["seed"], n_jobs=-1)
    forest.fit(X, y)
    return forest


def forest_key(settings):
    parts = []
    for name in FOREST_NAMES:
        parts.append(name + "=" + str(settings[name]))
    return ", ".join(parts)


def try_setting(settings, note=""):
    """One experiment: fit (or reuse) the forest, pool its range on the 2025 validation hours, score it, log it."""
    key = forest_key(settings)
    if key not in forests:
        started = time.time()
        X, y = training_table(settings["trained_on"], X_all.index[in_train])
        forest = fit_forest(settings, X, y)
        seconds = time.time() - started
        joblib.dump(forest, "trial.joblib", compress=3)
        shipped_mb = os.path.getsize("trial.joblib") / 1e6 * len(X_all) / len(X)
        forests[key] = {"q": percentiles_on_runs(forest, valid_inputs), "mb": shipped_mb, "seconds": seconds}
    saved = forests[key]
    low, expected, high = pooled(saved["q"], settings["hrrr_runs"], settings["range_widen"])
    ok = ~np.isnan(expected)
    row = {"experiment": len(EXPERIMENTS) + 1}
    for name in SETTING_NAMES:
        row[name] = settings[name]
    row.update(score_range(low[ok], expected[ok], high[ok], y_valid.values[ok]))
    row["shipped file (MB)"] = round(saved["mb"], 1)
    row["fits budget"] = saved["mb"] <= BUDGET_MB
    row["fit (s)"] = round(saved["seconds"])
    row["note"] = note
    EXPERIMENTS.append(row)
    print("#" + str(row["experiment"]), note, "| CRPS", row["CRPS (MW)"], "| inside", row["inside range (%)"], "% | width",
          row["average width (MW)"], "MW |", row["shipped file (MB)"], "MB")
    return row


def show_log(top=None):
    """The experiment log, best CRPS first."""
    log = pd.DataFrame(EXPERIMENTS).sort_values("CRPS (MW)").set_index("experiment")
    if top is not None:
        log = log.head(top)
    return log


def settings_of(experiment):
    """The settings dictionary of one experiment in the log, ready to copy and change."""
    row = EXPERIMENTS[experiment - 1]
    settings = {}
    for name in SETTING_NAMES:
        settings[name] = row[name]
    return settings


def best_so_far():
    """The settings of the lowest-CRPS experiment that fits the budget."""
    log = show_log()
    log = log[log["fits budget"]]
    return settings_of(int(log.index[0]))


print("The machine is ready. Log so far:", len(EXPERIMENTS), "experiments")

**Experiment #1 is always the setting you shipped in 6C**, so every later experiment has something to beat. Type
the `hrrr_runs` your 6C Part 4 picked (look at your model card) if it wasn't 4.

In [ ]:
BASE = {"min_samples_leaf": 10, "max_features": 1.0, "n_estimators": 100, "seed": SEED,
        "trained_on": "observed weather", "hrrr_runs": 4, "range_widen": 1.0}
try_setting(BASE, "6C as shipped")
show_log()

**How to run your own experiment** (you'll do this a lot tonight): copy a settings dictionary, change one thing,
call `try_setting`.

```python
trial = dict(BASE)                      # a copy, so BASE stays the same
trial["min_samples_leaf"] = 5           # the one thing you change
try_setting(trial, "smaller leaves")    # a short note you'll understand tomorrow
```

Change **one thing at a time** while you explore. Change two at once and you can't tell which one helped.

# Part 3 · Experiment 1: leaf size (10 min)
`min_samples_leaf` is the smallest number of training hours allowed in a leaf. A quantile forest makes its range from
the training loads that land in the same leaves as the new hour (6B). So:

- **Big leaves** (50 hours each): every leaf mixes many different days, so the range is **wide**.
- **Small leaves** (1 or 2 hours): each leaf holds only very similar hours, so the range is **narrow**, maybe too narrow.

Somewhere in between is the lowest CRPS. Find it.

In [ ]:
LEAF_SIZES = [1, 2, 5, 10, 20, 50]
# YOUR CODE HERE: one experiment for each leaf size in LEAF_SIZES, everything else the same as BASE.
# Start every note with "Experiment 1" so the plot below finds them, e.g. "Experiment 1: leaf 5".
# Hint: a for loop over LEAF_SIZES; inside it, copy BASE, change "min_samples_leaf", call try_setting.
# Each forest takes 10 to 30 seconds in Colab.


In [ ]:
def plot_experiment(prefix, knob):
    """CRPS, coverage and width against one knob, for every experiment whose note starts with prefix."""
    log = pd.DataFrame(EXPERIMENTS)
    picked = log[log["note"].str.startswith(prefix)].sort_values(knob)
    if len(picked) == 0:
        print("No experiments with a note starting", repr(prefix), "yet: run the cell above first.")
        return
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    columns = ["CRPS (MW)", "inside range (%)", "average width (MW)"]
    for i in range(3):
        axes[i].plot(picked[knob].astype(str), picked[columns[i]], marker="o")
        axes[i].set_xlabel(knob)
        axes[i].set_title(columns[i])
    axes[1].axhline(80, color="gray", linestyle="--", linewidth=1)
    axes[0].axhline(EXPERIMENTS[0]["CRPS (MW)"], color="gray", linestyle="--", linewidth=1, label="6C as shipped")
    axes[0].legend(fontsize=8)
    plt.tight_layout()
    plt.show()


plot_experiment("Experiment 1", "min_samples_leaf")
show_log(top=5)

**Read it:** as the leaves shrink, what happens to the width? To the share inside the range? Does CRPS keep
falling all the way down to a leaf of 1, or does it turn around? And look at the file size column: which leaf sizes
would you be **allowed** to ship?

# Part 4 · Experiment 2: features per split (5 min)
`max_features` is the share of the 11 features each split is allowed to look at. `1.0` = all of them (6C). `0.5` = a
random half at every split. Less choice per split makes the trees **more different from each other**, and a forest of
different trees often averages better. It also changes the range, because the leaves change.

Start from your best setting so far (`best_so_far()` does that for you) and try `max_features` = 1.0, 0.7, 0.5 and 0.33.

In [ ]:
start = best_so_far()
print("Starting from:", start)
# YOUR CODE HERE: from start, try max_features = 1.0, 0.7, 0.5 and 0.33, one experiment each.
# Start every note with "Experiment 2". (One of them may be a forest you already fit: that one is instant.)


In [ ]:
plot_experiment("Experiment 2", "max_features")
show_log(top=5)

# Part 5 · Experiment 3: two knobs at once (10 min)
Leaf size and `max_features` work together: the best leaf with all 11 features isn't always the best leaf with half
of them. A **grid** tries every pair. Twelve pairs here: about 4 minutes in Colab (pairs you already fit are instant).

In [ ]:
# YOUR CODE HERE: every pair of leaf size [2, 3, 5, 8] and max_features [0.33, 0.5, 0.7], starting from BASE.
# Start every note with "Experiment 3".
# Hint: two for loops, one inside the other. 4 x 3 = 12 experiments.


In [ ]:
log = pd.DataFrame(EXPERIMENTS)
grid = log[log["note"].str.startswith("Experiment 3")]
if len(grid) == 0:
    print("Run the grid above first.")
else:
    table = grid.pivot_table(index="min_samples_leaf", columns="max_features", values="CRPS (MW)", aggfunc="min")
    fig, ax = plt.subplots(figsize=(6, 4))
    image = ax.imshow(table.values, cmap="viridis_r")
    ax.set_xticks(range(len(table.columns)))
    ax.set_xticklabels(table.columns)
    ax.set_yticks(range(len(table.index)))
    ax.set_yticklabels(table.index)
    ax.set_xlabel("max_features")
    ax.set_ylabel("min_samples_leaf")
    for i in range(len(table.index)):
        for j in range(len(table.columns)):
            ax.text(j, i, table.values[i, j], ha="center", va="center", color="white", fontsize=9)
    ax.set_title("Validation CRPS (MW), lower is better")
    fig.colorbar(image)
    plt.show()
show_log(top=5)

**Read it:** is the best corner at the edge of the grid? If the lowest CRPS sits on an edge (the smallest leaf or
the smallest share you tried), the real best might be just past it: that's a reason to try one more step in Part 8.

# Part 6 · Experiment 4: the range knobs (5 min)
Two settings live in the **model card**, not in the forest, so they cost nothing to try (no refit):

- **`hrrr_runs`**: how many of the newest HRRR runs go in the pile (6C Part 4). More runs = more weather scenarios, but
  older runs are worse.
- **`range_widen`**: stretch low and high away from the expected value. `1.2` = 20% wider. It's how you fix a range
  that's honest about the weather but still a bit too narrow (6C found coverage a few points under 80%).

Sweep both on your best forest.

In [ ]:
start = best_so_far()
# YOUR CODE HERE: from start, every pair of hrrr_runs [1, 2, 4, 8] and range_widen [0.9, 1.0, 1.1, 1.2, 1.3].
# Start every note with "Experiment 4". No refits, so all 20 take a few seconds.


In [ ]:
log = pd.DataFrame(EXPERIMENTS)
sweep = log[log["note"].str.startswith("Experiment 4")]
if len(sweep) == 0:
    print("Run the sweep above first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    for runs in sorted(sweep["hrrr_runs"].unique()):
        line = sweep[sweep["hrrr_runs"] == runs].sort_values("range_widen")
        axes[0].plot(line["range_widen"], line["CRPS (MW)"], marker="o", label=str(runs) + " runs")
        axes[1].plot(line["range_widen"], line["inside range (%)"], marker="o", label=str(runs) + " runs")
    axes[0].set_title("CRPS (MW), lower is better")
    axes[1].set_title("Inside the range (%)")
    axes[1].axhline(80, color="gray", linestyle="--", linewidth=1)
    for ax in axes:
        ax.set_xlabel("range_widen")
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
show_log(top=5)

**Read it:** the widen with the **lowest CRPS** and the widen that puts coverage closest to 80% are usually close,
but not the same. CRPS is the score that counts. A range at 95% coverage is honest, but too wide to be useful, and
CRPS charges you for every extra MW of width.

# Part 7 · Is it real? Seed noise (5 min)
A random forest is random: a different `seed` grows different trees. So two settings can differ by 1 MW of CRPS just
by luck. Before you believe a gain, measure the luck: refit your best setting with three other seeds.

In [ ]:
best = best_so_far()
for seed in [1, 2, 3]:
    trial = dict(best)
    trial["seed"] = seed
    try_setting(trial, "Seed check: seed " + str(seed))

log = pd.DataFrame(EXPERIMENTS)
seeds = log[log["note"].str.startswith("Seed check")]["CRPS (MW)"]
noise = round(float(seeds.max() - seeds.min()), 1)
gain = round(EXPERIMENTS[0]["CRPS (MW)"] - float(seeds.mean()), 1)
print("Seed noise: CRPS moves by", noise, "MW just by changing the seed.")
print("Your gain over 6C as shipped:", gain, "MW (average over the 3 seeds).")
if gain > 2 * noise:
    print("The gain is more than twice the noise: it's real.")
else:
    print("The gain is within the noise: don't trust it yet. More trees (Part 8, Challenge A) make a forest less noisy.")

# Part 8 · Your turn: experiment (15+ min)
The machine is yours. Every call to `try_setting` goes in the log, and the log is your proof. At least **two**
challenges, more if you're curious. Start each note with "Challenge".

**Challenge A · More trees.** `n_estimators` = 50, 200, 300 on your best setting. More trees = less seed noise and
smoother percentiles, but a bigger file and a slower fit. Where does it stop paying off? Watch the budget.

**Challenge B · Train on forecast weather.** Every forest so far learned from the weather that **really happened**. In
production it only ever sees **forecasts**. Set `"trained_on"` to `"HRRR 12h forecast"`: the forest then learns from
the newest HRRR forecast issued 12 hours before each training hour, forecast mistakes and all. Does its range get more
honest? Does CRPS drop? (A big question in real forecasting. Either answer is a good answer, if you can show it.)

**Challenge C · Push past the edge.** If your Part 5 winner sat on the edge of the grid, try one step beyond it.

**Challenge D · Your own idea.** Anything `try_setting` can take. Write down why you think it will help **before** you
run it. Then see.

In [ ]:
# Challenge A, B, C or D: YOUR EXPERIMENTS HERE (one per line, or a loop). Start each note with "Challenge".


In [ ]:
# Another challenge:


In [ ]:
# Another challenge:


In [ ]:
show_log(top=10)

# Part 9 · The log and the winner (5 min)
Pick the winner with a **rule**, written down before you look: the **lowest validation CRPS that fits the budget**.
`best_so_far()` is that rule. If two settings are within the seed noise of each other, you may overrule it and pick
the **smaller file** (faster, cheaper, same score): type its experiment number in `WINNER_EXPERIMENT`.

Then save the log. You'll want it for the final project report: section 5 asks for one improvement you tested
fairly, with before and after numbers, and this log is that proof.

In [ ]:
WINNER_EXPERIMENT = None                  # None = the rule (lowest CRPS within budget); or type an experiment number

log = show_log()
if WINNER_EXPERIMENT is None:
    WINNER_EXPERIMENT = int(log[log["fits budget"]].index[0])
WINNER = settings_of(WINNER_EXPERIMENT)
winner_valid_crps = float(log.loc[WINNER_EXPERIMENT, "CRPS (MW)"])
print("6C as shipped:", BASE, "| validation CRPS", EXPERIMENTS[0]["CRPS (MW)"])
print("Your winner:  ", WINNER, "| validation CRPS", winner_valid_crps, "(experiment #" + str(WINNER_EXPERIMENT) + ")")

log.to_csv("lab6d_experiment_log.csv")
print(len(log), "experiments saved to lab6d_experiment_log.csv (Files panel on the left: right-click, Download)")
show_log(top=10)

# Part 10 · One look at 2026 (5 min)
Now, and only now, the test year. Both forests are refit on **2023 to 2025** (everything before the test), then
judged on **every hour** of 2026 at 1, 12 and 24 hours ahead, pooled exactly like production. This is your
**before and after**. Whatever it says, you keep the winner: you don't go back and tune on what you see here.

In [ ]:
test_hours = X_all.index[(years == TEST_YEAR) & (X_all.index <= archive["hour_utc"].max())]
y_test = y_all.loc[test_hours]
before_test = X_all.index[years < TEST_YEAR]
one_look = []
for name in ["6C as shipped", "your winner"]:
    settings = BASE
    if name == "your winner":
        settings = WINNER
    started = time.time()
    X, y = training_table(settings["trained_on"], before_test)
    forest = fit_forest(settings, X, y)
    for ahead in [1, 12, 24]:
        runs = hrrr_runs_as_issued(test_hours, ahead)
        inputs = []
        for k in range(N_RUNS):
            inputs.append(make_features(test_hours, runs[k], df["load_mw"]))
        low, expected, high = pooled(percentiles_on_runs(forest, inputs), settings["hrrr_runs"], settings["range_widen"])
        ok = ~np.isnan(expected)
        row = {"forest": name, "lead": str(ahead) + " h ahead"}
        row.update(score_range(low[ok], expected[ok], high[ok], y_test.values[ok]))
        one_look.append(row)
    print(name, "done in", round(time.time() - started), "seconds")
one_look = pd.DataFrame(one_look)
one_look.pivot(index="forest", columns="lead", values=["CRPS (MW)", "inside range (%)", "average width (MW)"])

**Read it:** did the winner beat 6C's setting on 2026 too? By about as much as on 2025, or less? (A bit less is
normal: some of every validation win is luck.) If it lost, say so in your answers. That's an honest result.

# Part 11 · Ship it (10 min)
The winner is refit on **all** the data (2023 to October 15, 2026), like every model you've shipped. The model card
carries your tuned `hrrr_runs` and `range_widen` to `forecast-24h`. Your 6C forest stays in
`energy/models/history/` in case you want it back.

In [ ]:
started = time.time()
X, y = training_table(WINNER["trained_on"], X_all.index)
final_qrf = fit_forest(WINNER, X, y)
joblib.dump(final_qrf, "model_qrf.joblib", compress=3)
size_mb = round(os.path.getsize("model_qrf.joblib") / 1e6, 1)
print("Refit on", len(X), "hours in", round(time.time() - started), "seconds:", size_mb, "MB")
if size_mb > BUDGET_MB:
    print("Over the", BUDGET_MB, "MB budget: pick a setting with bigger leaves or fewer trees (Part 9), then re-run.")

winner_12 = one_look[(one_look["forest"] == "your winner") & (one_look["lead"] == "12 h ahead")].iloc[0]
card = {
    "model_type": "RandomForestQuantileRegressor",
    "best_params": {"n_estimators": str(WINNER["n_estimators"]), "min_samples_leaf": str(WINNER["min_samples_leaf"]),
                    "max_features": str(WINNER["max_features"]), "trained_on": WINNER["trained_on"]},
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "range_method": "hrrr_qrf_pooled",
    "range_quantiles": [0.1, 0.9],
    "hrrr_runs": int(WINNER["hrrr_runs"]),
    "range_widen": float(WINNER["range_widen"]),
    "tuning": {"experiments": len(EXPERIMENTS), "validation_year": VALID_YEAR,
               "validation_crps_mw_12h": winner_valid_crps, "validation_crps_mw_12h_6c": EXPERIMENTS[0]["CRPS (MW)"]},
    "test_inside_pct_12h": float(winner_12["inside range (%)"]),
    "test_width_mw_12h": float(winner_12["average width (MW)"]),
    "test_crps_mw_12h": float(winner_12["CRPS (MW)"]),
    "train_period": "2023-01-01 to 2026-10-15 (Connecticut time)",
    "seed": int(WINNER["seed"]),
}
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=2)
card

Upload and run `forecast-24h` now (the same checks as 6C Part 5b). When the **"Allow access"** pop-up opens, pick
the Gmail you used for Google Cloud.

In [ ]:
UPLOAD = "YOUR-ENERGY-PROJECT-ID" not in PROJECT_ID and "YOUR-ENERGY-PROJECT-ID" not in BUCKET_NAME
if size_mb > BUDGET_MB:
    UPLOAD = False
    print("Not uploading: the model is over the budget (see above).")
latest_blob = None
if not UPLOAD:
    print("DEMO MODE: type your PROJECT_ID and BUCKET_NAME in Part 0 to ship. Nothing uploaded.")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    account_lines = !gcloud config get-value account
    print("1. Signed in as:", account_lines[-1].strip() if len(account_lines) > 0 else "unknown")
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    if not bucket.exists():
        raise RuntimeError("Can't see the bucket " + BUCKET_NAME + ": check Part 0, or Runtime > Disconnect and delete "
                           "runtime and pick your Google Cloud Gmail in the pop-up.")
    print("2. Bucket found:", BUCKET_NAME)
    state_lines = !gcloud functions describe forecast-24h --gen2 --region=us-central1 --project={PROJECT_ID} --format="value(state)"
    print("3. forecast-24h:", state_lines[-1].strip() if len(state_lines) > 0 else "NOT FOUND")

    stamp = card["trained_at"].replace(":", "-")
    for name in ["model_qrf.joblib", "model_card.json"]:
        bucket.blob("energy/models/" + name).upload_from_filename(name)
        bucket.blob("energy/models/history/" + name.replace(".", "_" + stamp + ".", 1)).upload_from_filename(name)
        print("Live:", "gs://" + BUCKET_NAME + "/energy/models/" + name)

    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(16):                 # every 15 seconds, for up to 4 minutes
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if not ready:
        print("No new forecast after 4 minutes. Run the log cell in 6C Part 5b and paste the log into Claude.")
    else:
        mine = pd.read_csv(io.BytesIO(latest_blob.download_as_bytes()))
        method = str(mine["range_method"].iloc[0])
        width = round(float((mine["predicted_high_mw"] - mine["predicted_low_mw"]).mean()))
        if method.startswith("hrrr_qrf_pooled"):
            print("Tuned range is live:", NETID.strip().lower(), "| leaf", WINNER["min_samples_leaf"], "| max_features",
                  WINNER["max_features"], "|", WINNER["n_estimators"], "trees |", WINNER["hrrr_runs"], "runs | widen",
                  WINNER["range_widen"], "| average width", width, "MW")
        else:
            print("range_method came back as", method, "- do 6C Part 0 (Sync fork) and re-run this cell.")

# Part 12 · Your dashboard on Cloud Run (10 min)
Your model has been forecasting every hour since Lab 5, but the only place you can see how it's doing is the class
leaderboard. Now you deploy **your own** public web page on Google Cloud Run, in your energy project. It shows:

- your next 24 hours (low, expected and high) next to ISO New England's forecast
- your live score, worked out the same way as the leaderboard (CRPS, skill, coverage)
- **My models**: every model you ever shipped (Lab 5, 6C, 6D), when it went live, and how each one scored live.
  Shipping a new model never deletes the old one: every model and every forecast stays in your bucket.

It's the same deployment pattern as your Cloud Functions: GitHub Actions builds a **container** (a small Linux box
with Python, Streamlit and the app, described by `dashboard/Dockerfile`), pushes it to **Artifact Registry** in your
project, and **Cloud Run** runs it as a web page that scales to zero when nobody's looking (inside the free tier).
It runs as your project's runtime service account and only *reads* your bucket, so there is no key to make or leak.

**Four steps, all on GitHub:**
1. Open your `energy-forecasting` fork and click **Sync fork → Update branch** (this brings in `dashboard/` and the
   workflow **Deploy dashboard (Cloud Run)**).
2. **Settings → Secrets and variables → Actions → Variables tab → New repository variable**:
   name `DASHBOARD_NICKNAME`, value = your leaderboard nickname. **Not** your name or NetID: the page is public.
3. **Actions → Deploy dashboard (Cloud Run) → Run workflow → Run workflow**. Wait about 3 minutes for the green check.
4. Run the cell below. It finds your page, checks it answers, and prints the line for your submission.

If the workflow fails at *Pick an image folder*, run the one command it prints in **Cloud Shell** (your energy
project), then run the workflow again. If it fails anywhere else, open the red step and paste the error into Claude
or Gemini with "I'm deploying a Streamlit app to Cloud Run from GitHub Actions".

**Make it yours (optional):** change colors or add a chart in `dashboard/app.py` right on GitHub (pencil icon) and
commit: the page redeploys by itself in about 3 minutes.

In [ ]:
import time
import urllib.request

dashboard_url = ""
if "YOUR-ENERGY-PROJECT-ID" in PROJECT_ID:
    print("DEMO MODE: type your PROJECT_ID in Part 0 first.")
else:
    from google.colab import auth
    auth.authenticate_user()
    url_lines = !gcloud run services describe energy-dashboard --region=us-central1 --project={PROJECT_ID} --format="value(status.url)" 2>&1
    found = [line.strip() for line in url_lines if line.strip().startswith("https://")]
    if len(found) == 0:
        print("No energy-dashboard service in " + PROJECT_ID + " yet. Do steps 1 to 3 above, wait for the green check,")
        print("then re-run this cell. (gcloud said: " + " ".join(url_lines)[-300:] + ")")
    else:
        dashboard_url = found[0]
        print("Found:", dashboard_url, "- waking it up", end="")
        ok = False
        for attempt in range(12):                  # a page that scaled to zero takes a few seconds to wake up
            try:
                with urllib.request.urlopen(dashboard_url + "/_stcore/health", timeout=30) as response:
                    ok = response.status == 200
            except Exception:
                ok = False
            if ok:
                break
            print(".", end="")
            time.sleep(5)
        print()
        if ok:
            print("Dashboard is live:", dashboard_url, "(" + PROJECT_ID + ")")
            print("Open it, look at your My models table, and paste the line above in your submission.")
        else:
            print("The service exists but did not answer. Look at its logs:")
            print("  gcloud run services logs read energy-dashboard --region=us-central1 --project=" + PROJECT_ID + " --limit=30")

# Part 13 · Questions and submit (5 min)
13. **Part 3:** As `min_samples_leaf` got smaller, what happened to the width, the coverage and the CRPS? Why does a
    smaller leaf make a narrower range?
14. **Parts 4 and 5:** What were your best `min_samples_leaf` and `max_features`, and how much lower was their
    validation CRPS than 6C's setting? Was the best pair on the edge of your grid?
15. **Parts 6 and 7:** Which `hrrr_runs` and `range_widen` did you pick, and what was the coverage? Was your gain over
    6C bigger than the seed noise?
16. **Part 8:** Describe your two challenges: what you expected, what happened, and the numbers. Did either one win?
17. **Part 10:** Your before and after on 2026, 12 hours ahead (CRPS, coverage, width). Why do we look at 2026 only
    once, and only at the end?
18. **Part 12:** Paste your dashboard link. Which of your models in the **My models** table has the best live CRPS
    skill so far, and does that match what your notebooks predicted (the notebook test score)?

## Submit Lab 6
In the **Submit Lab 6** text box on HuskyCT, paste:
1. the **"Tuned range is live"** line Part 11 printed (or 6C's "Range is live" line if you kept the 6C forest),
2. the **"Dashboard is live"** line Part 12 printed, then
3. your answers to questions **1 to 18** (1-4 in 6A, 5-8 in 6B, 9-12 in 6C, 13-18 here), numbered.

Due **Friday, November 20, 11:59 PM** (Hartford and Stamford). Your live CRPS from **Saturday, November 21** is your
final project score: watch your coverage on the leaderboard for a day or two, and if it's far from 80%, change only
`range_widen` in the model card and upload the card.

Keep `lab6d_experiment_log.csv`. It's your evidence for the final project report.

In [ ]:
if len(EXPERIMENTS) > 0:
    print("Experiments run:", len(EXPERIMENTS))
    print(show_log(top=3)[["min_samples_leaf", "max_features", "n_estimators", "trained_on", "hrrr_runs", "range_widen",
                          "CRPS (MW)", "inside range (%)", "average width (MW)", "shipped file (MB)"]])

---

**Data credits:** weather from NOAA's HRRR via dynamical.org; load from ISO New England. quantile-forest by Zillow
(github.com/zillow/quantile-forest).